## Pred. #3 - Layered builds of the baseline AME figures

This notebook regenerates the four-panel `AME_full_is_ai_*` figures from
`onet_antrhopicIndex_execTypeVaryingDWA.ipynb` in four layered build
stages, each at the same figure size, labels and axis ranges as the
originals so they can be dropped into different builds of the same slide:

1. **`empty`** - axes, panel titles, axis labels, x/y ranges, zero line. No observed value, no histogram, no legend.
2. **`randomization`** - empty layout plus the placebo (task-position-reshuffled) histogram. No observed line.
3. **`observed`** - empty layout plus the red dashed observed AME line, its 1.645-SE band, and the legend.
4. **`observed_hist`** - the observed line plus the placebo histogram (matches the original figure).

The notebook reads the cached AME CSVs that the main DWA notebook wrote
(`regression_summaries_is_ai/regression_ame_results_full_*.csv`); it does
not re-run any regressions.

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
# --- Match the configuration of onet_antrhopicIndex_execTypeVaryingDWA.ipynb ---
DEPENDENT_VAR = 'is_ai'
KEEP_RANDOM_DWAS = False
N_SHUFFLES = 1000

TARGET_REGS = ['prev2_is_ai', 'prev_is_ai', 'next_is_ai', 'next2_is_ai']
SPECS = ['no_fe_no_dwa', 'major_fe_no_dwa', 'minor_fe_no_dwa',
         'no_fe_with_dwa', 'no_fe_no_dwa_withTaskDWACount',
         'no_fe_with_dwa_withTaskDWACount']
PLOT_TITLES = ['Task Before Previous Task', 'Previous Task',
               'Next Task', 'Task After Next Task']
VAR_LABELS = {
    'prev2_is_ai': '($t-2$) Task AI',
    'prev_is_ai':  '($t-1$) Task AI',
    'next_is_ai':  '($t+1$) Task AI',
    'next2_is_ai': '($t+2$) Task AI',
}

path_suffix = '' if KEEP_RANDOM_DWAS else '_noTasksWithRepetitiveDWAs'
main_folder_path = '..'
input_data_path = f'{main_folder_path}/data'
regression_dir = (
    f'{input_data_path}/computed_objects/'
    f'execTypeVaryingDWA_anthropicIndex{path_suffix}/'
    f'regression_summaries_{DEPENDENT_VAR}'
)
output_plot_path = (
    f'{main_folder_path}/writeup/plots/'
    f'execTypeVaryingDWA{path_suffix}/{DEPENDENT_VAR}/builds'
)
Path(output_plot_path).mkdir(parents=True, exist_ok=True)

# Mirror the figure size used for the per-spec row plots in the main notebook.
ROW_FIGSIZE = (24, 5)
BINS = 30

In [3]:
def results_to_dict(df_results):
    """Convert a tidy AME-results DataFrame into nested dicts keyed by
    (spec, term). Returns (coef_dict, se_dict)."""
    coef_out = {spec: {term: np.nan for term in TARGET_REGS} for spec in SPECS}
    se_out   = {spec: {term: np.nan for term in TARGET_REGS} for spec in SPECS}
    if df_results is None or df_results.empty:
        return coef_out, se_out
    for _, row in df_results.iterrows():
        spec = row.get('model')
        term = row.get('term')
        if spec in coef_out and term in coef_out[spec]:
            if pd.notna(row.get('ame_coef')):
                coef_out[spec][term] = row['ame_coef']
            if pd.notna(row.get('ame_se')):
                se_out[spec][term] = row['ame_se']
    return coef_out, se_out


# `regression_ame_results_full_0.csv` is what the main notebook treats as
# the OBSERVED estimate (it gets written by the non-shuffled regression run
# in cell 18 of the main notebook).
obs_csv = Path(regression_dir) / f'regression_ame_results_full_0.csv'
if not obs_csv.exists():
    raise FileNotFoundError(
        f'Observed AME file not found: {obs_csv}. Run the main DWA '
        f'notebook first to populate the cached results.'
    )
obs_df = pd.read_csv(obs_csv)
obs_coef, obs_se = results_to_dict(obs_df)

# Placebo distribution: shuffles 1..N_SHUFFLES (skip i=0 because that file
# carries the observed run, not a shuffle).
resh = {spec: {t: [] for t in TARGET_REGS} for spec in SPECS}
missing = []
for i in range(1, N_SHUFFLES + 1):
    f = Path(regression_dir) / f'regression_ame_results_full_{i}.csv'
    if not f.exists():
        missing.append(i)
        continue
    df = pd.read_csv(f)
    c, _ = results_to_dict(df)
    for spec in SPECS:
        for term in TARGET_REGS:
            resh[spec][term].append(c[spec][term])

n_loaded = len(resh[SPECS[0]][TARGET_REGS[0]])
print(f'Loaded {n_loaded} placebo runs (missing: {len(missing)} of {N_SHUFFLES}).')

Loaded 999 placebo runs (missing: 1 of 1000).


In [4]:
# Pre-compute axis ranges so all three builds of a given spec/term share
# identical x and y limits (and the empty build inherits them too).
#
# X-limits: symmetric around 0, covering the full union of observed and
# placebo AMEs across ALL specs and terms - exactly the rule used in the
# main notebook so that builds align with the existing figures.
all_vals = []
for spec in SPECS:
    for term in TARGET_REGS:
        for v in resh[spec][term]:
            if not np.isnan(v):
                all_vals.append(v)
        v = obs_coef[spec][term]
        if not np.isnan(v):
            all_vals.append(v)
if not all_vals:
    raise RuntimeError('No AME values loaded; aborting.')

g_bound = max(abs(min(all_vals)), abs(max(all_vals)))
span = 2 * g_bound
X_LIM = (-g_bound - span * 0.01, g_bound + span * 0.01)

# Y-limits: per (spec, term) panel, the natural histogram height plus 5%.
# This guarantees the empty/observed builds have identical y-range to the
# observed_hist build, so the legend and lines sit in the same place.
y_max_panel = {}
for spec in SPECS:
    for term in TARGET_REGS:
        vals = np.array(resh[spec][term], dtype=float)
        vals = vals[~np.isnan(vals)]
        if len(vals):
            counts, _ = np.histogram(vals, bins=BINS)
            y_max_panel[(spec, term)] = max(counts.max() * 1.05, 1.0)
        else:
            y_max_panel[(spec, term)] = 1.0

print(f'Shared x-limits: ({X_LIM[0]:.4f}, {X_LIM[1]:.4f})')

Shared x-limits: (-0.1587, 0.1587)


In [5]:
BUILDS = ('empty', 'randomization', 'observed', 'observed_hist')


def render_row(spec, build, out_path):
    """Render one 4-panel row for a given spec in one of four build modes.

    Modes:
      'empty'         - axes, titles, labels, x/y limits, zero reference line.
      'randomization'    - empty + placebo histogram only (no observed line/CI).
      'observed'      - empty + red dashed observed AME + 1.645*SE band + legend.
      'observed_hist' - observed line + placebo histogram + legend.
    """
    if build not in BUILDS:
        raise ValueError(f'Unknown build: {build}')

    color_row = plt.cm.tab10(SPECS.index(spec) % 10)

    fig, axs = plt.subplots(
        nrows=1, ncols=len(TARGET_REGS),
        figsize=ROW_FIGSIZE, sharey=False,
    )
    if len(TARGET_REGS) == 1:
        axs = [axs]

    show_hist = build in ('randomization', 'observed_hist')
    show_obs  = build in ('observed', 'observed_hist')

    for c, term in enumerate(TARGET_REGS):
        ax = axs[c]

        # Histogram (in randomized + observed_hist builds)
        vals = np.array(resh[spec][term], dtype=float)
        vals_clean = vals[~np.isnan(vals)]
        if show_hist and len(vals_clean):
            # Match the original figure: do NOT force `range=` on hist,
            # so the BINS auto-fit to the data span and bars stay narrow.
            ax.hist(
                vals_clean, bins=BINS,
                color=color_row, alpha=0.7, edgecolor='k',
                label='Task Position Reshuffled AMEs', zorder=2,
            )

        # Observed line + CI band (in observed + observed_hist builds)
        obs_val = obs_coef.get(spec, {}).get(term, np.nan)
        if show_obs and not np.isnan(obs_val):
            obs_se_v = obs_se.get(spec, {}).get(term, np.nan)
            if build in ('observed', 'observed_hist') and not np.isnan(obs_se_v):
                # 90% CI band (z = 1.645) - matches the original figure.
                se_band = 1.645 * obs_se_v
                ax.axvspan(
                    obs_val - se_band, obs_val + se_band,
                    color='red', alpha=0.08, zorder=1,
                )
                ax.axvline(obs_val - se_band, color='red', linestyle='--',
                           linewidth=1, alpha=0.9, zorder=3)
                ax.axvline(obs_val + se_band, color='red', linestyle='--',
                           linewidth=1, alpha=0.9, zorder=3)
            ax.axvline(
                obs_val, color='red', linestyle='--', linewidth=3,
                label=f'Observed = {obs_val:.3f}', zorder=4,
            )

        # Zero reference line - kept across all builds so the panel scaffolding
        # is identical regardless of which layer we are showing.
        ax.axvline(0.0, color='black', linestyle='-', linewidth=1.5,
                   alpha=0.5, zorder=4)

        # Titles / labels (always present)
        ax.set_title(VAR_LABELS.get(term, term), fontsize=15)
        if c == 0:
            ax.set_ylabel('Count', fontsize=15)
        ax.set_xlabel('Average Marginal Effect', fontsize=15)
        ax.grid(axis='y', linestyle=':', alpha=0.5)
        ax.set_xlim(*X_LIM)
        ax.set_ylim(0, y_max_panel[(spec, term)])

        # Legend only when there is something labelled (i.e., not 'empty').
        if build != 'empty':
            ax.legend(loc='best', fontsize=10)

    fig.tight_layout()
    fig.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close(fig)

In [6]:
# Generate all three builds for each spec.
for spec in SPECS:
    for build in BUILDS:
        out_name = f'AME_full_{DEPENDENT_VAR}_{spec}_build_{build}.png'
        out_path = f'{output_plot_path}/{out_name}'
        render_row(spec, build, out_path)
        print(f'Saved {out_path}')

Saved ../writeup/plots/execTypeVaryingDWA_noTasksWithRepetitiveDWAs/is_ai/builds/AME_full_is_ai_no_fe_no_dwa_build_empty.png


Saved ../writeup/plots/execTypeVaryingDWA_noTasksWithRepetitiveDWAs/is_ai/builds/AME_full_is_ai_no_fe_no_dwa_build_randomization.png


Saved ../writeup/plots/execTypeVaryingDWA_noTasksWithRepetitiveDWAs/is_ai/builds/AME_full_is_ai_no_fe_no_dwa_build_observed.png


Saved ../writeup/plots/execTypeVaryingDWA_noTasksWithRepetitiveDWAs/is_ai/builds/AME_full_is_ai_no_fe_no_dwa_build_observed_hist.png


Saved ../writeup/plots/execTypeVaryingDWA_noTasksWithRepetitiveDWAs/is_ai/builds/AME_full_is_ai_major_fe_no_dwa_build_empty.png


...

Saved ../writeup/plots/execTypeVaryingDWA_noTasksWithRepetitiveDWAs/is_ai/builds/AME_full_is_ai_no_fe_no_dwa_withTaskDWACount_build_observed_hist.png


Saved ../writeup/plots/execTypeVaryingDWA_noTasksWithRepetitiveDWAs/is_ai/builds/AME_full_is_ai_no_fe_with_dwa_withTaskDWACount_build_empty.png


Saved ../writeup/plots/execTypeVaryingDWA_noTasksWithRepetitiveDWAs/is_ai/builds/AME_full_is_ai_no_fe_with_dwa_withTaskDWACount_build_randomization.png


Saved ../writeup/plots/execTypeVaryingDWA_noTasksWithRepetitiveDWAs/is_ai/builds/AME_full_is_ai_no_fe_with_dwa_withTaskDWACount_build_observed.png


Saved ../writeup/plots/execTypeVaryingDWA_noTasksWithRepetitiveDWAs/is_ai/builds/AME_full_is_ai_no_fe_with_dwa_withTaskDWACount_build_observed_hist.png
